
# Check your setup: Windows

**Run this after the install, and before the session.** It checks, one thing at a time, that
everything the three workshop notebooks need is installed and working on this machine. It installs
nothing and changes nothing. It only looks and reports.

Do the install in `README.md` first. There is nothing here to check until the environment exists.
If this file reached you on its own, ahead of the workshop notebooks, it will say so and check the
environment anyway.

Run every cell in order with **Shift+Enter**, then read the last cell. Each failing check prints
the command that fixes it.

The TensorFlow check is the slow one. Everything else is close to instant.

> This is the Windows version. On macOS, use `00_check_your_setup_mac.ipynb` instead. It runs
the same checks. Only the commands and the Python versions are different.

## The checks

The first cell sets up how each result gets printed. Run it before the others.

In [ ]:
results = {}

def record(name, ok, detail, fix=""):
    """ok is True to pass, False to fail, or None for something worth reporting that is
    not a problem here."""
    results[name] = (ok, detail, fix)
    print(f"{'PASS' if ok else 'NOTE' if ok is None else 'FAIL'}  {name:28s} {detail}")
    if ok is False and fix:
        print(f"      fix: {fix}")

print("Checks will run one per cell. Read the last cell for the verdict.\n")

In [ ]:
# 1. Which Python is this notebook actually using?
import platform
import sys

v = sys.version_info
ok = (3, 10) <= v < (3, 11)
record("python version", ok, f"{v.major}.{v.minor}.{v.micro} on {platform.system()}",
       "on Windows das needs Python 3.10 exactly. Its TensorFlow 2.10 pin has no wheel for 3.11 "
       "or newer. If you meant to use the DAS environment, this is the wrong kernel. If not, "
       "recreate it with python=3.10, following the install steps in README.md")
print(f"\n      this kernel runs: {sys.executable}")
print("      That path has to be inside the environment you installed for the workshop.")
print("      If it is not, use Kernel -> Change Kernel... and pick the one that is.")

In [ ]:
# 2. The packages the notebooks need. numpy, scipy and matplotlib are used everywhere.
#      pandas is only used by part 3, which reads the CSVs you export from the GUI in part 2.
import importlib

for name, pip_name in [("numpy", "numpy"), ("scipy", "scipy"),
                       ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    try:
        m = importlib.import_module(name)
        record(name, True, getattr(m, "__version__", "installed"))
    except ImportError:
        record(name, False, "not installed", f"pip install {pip_name}")
    except Exception as exc:
        record(name, False, f"installed but will not import: {type(exc).__name__}",
               "the installed packages do not match each other. Recreate the environment "
               "following README.md")

In [ ]:
# 3. The clustering extras. Only part 1's last section needs these.
#      A copy built against a different numpy raises something other than ImportError, so both
#      are caught here. "Installed but will not import" is a different problem from "not
#      installed", and if it were not caught it would stop the notebook instead of showing a FAIL.
for name, pip_name in [("umap", "umap-learn"), ("hdbscan", "hdbscan")]:
    try:
        m = importlib.import_module(name)
        record(name, True, getattr(m, "__version__", "installed"))
    except ImportError:
        record(name, False, "not installed (optional)", f"pip install {pip_name}")
    except Exception as exc:
        record(name, False, f"installed but will not import: {type(exc).__name__}",
               f"pip install --force-reinstall {pip_name}")

In [ ]:
# 4. DAS itself. Parts 2 and 3 need it. Part 1 does not.
#      The das package holds nothing but a version string, so passing here only means it is
#      installed. Checks 5 and 6 test whether it can actually train and open its window.
try:
    import das
    record("das", True, getattr(das, "__version__", "installed"))
except Exception as exc:
    record("das", False, f"not installed or will not import ({type(exc).__name__})",
           "follow the install steps in README.md")

In [ ]:
# 5. TensorFlow, which DAS trains on. Importing it is not enough to prove it works, so this
#      trains a throwaway model for one step. That is what catches a half-broken install.
import os
import warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
try:
    import numpy as np
    import tensorflow as tf
    model = tf.keras.Sequential([tf.keras.layers.Dense(2, input_shape=(3,))])
    model.compile(optimizer="adam", loss="mse")
    model.fit(np.zeros((4, 3)), np.zeros((4, 2)), epochs=1, verbose=0)
    record("tensorflow", True, f"{tf.__version__}, and a one-step fit ran")
except ImportError:
    record("tensorflow", False, "not installed (needed for parts 2 and 3)",
           "follow the install steps in README.md")
except Exception as exc:
    record("tensorflow", False, f"imports but will not train: {type(exc).__name__}",
           "recreate the das environment following README.md")

In [ ]:
# 6. The window part 2 opens. A Qt binding on its own is not enough. `das gui` loads the
#      annotation app, which comes from xarray-behave, and that is the package you have to
#      install by name, so it is an easy one to forget. This imports what the GUI imports.
#      Check 5 has already started TensorFlow, so importing das here prints a "physical devices
#      cannot be modified" error. You never see it when you run `das gui` for real, because that
#      starts a fresh process. It only shows up because of the order these checks run in, so the
#      lines below hide it.
import logging

das_log = logging.getLogger("das.train")
was_level = das_log.level
das_log.setLevel(logging.CRITICAL)
try:
    import qtpy
    from qtpy.QtCore import qVersion
    import pyqtgraph
    import xarray_behave
    import xarray_behave.gui.app          # importing it does not open a window
    record("gui toolkit", True,
           f"xarray-behave {xarray_behave.__version__}, pyqtgraph {pyqtgraph.__version__}, "
           f"qtpy {qtpy.__version__}, Qt {qVersion()}")
except Exception as exc:
    record("gui toolkit", False, f"unavailable ({type(exc).__name__}: {exc})",
           'pip install "xarray-behave==0.37.4", or follow the install steps in README.md')
finally:
    das_log.setLevel(was_level)

In [ ]:
# 7. Can this notebook play sound? The tone is 150 Hz, the same pitch as sine song, and the
#      hardest thing in the workshop to hear on laptop speakers.
try:
    from IPython.display import Audio, display
    import numpy as np
    fs = 10_000
    t = np.arange(0, 2.0, 1 / fs)
    tone = 0.3 * np.sin(2 * np.pi * 150 * t)
    display(Audio(tone, rate=fs))
    record("audio widget", True, "a player should appear just above. Press play")
    print("      You should hear a low hum for two seconds. If the player is there but you")
    print("      hear nothing, the problem is your speakers, not your setup. Try headphones.")
except Exception as exc:
    record("audio widget", False, f"did not render ({type(exc).__name__})",
           "open the notebook in JupyterLab in a browser rather than inside an editor")

In [ ]:
# 8. The workshop notebooks, and a folder you can write to. The three parts may not be here
#      yet, because this check goes out before the rest of the folder does. Their absence says
#      nothing about the environment, so it is reported as a note rather than a failure.
from pathlib import Path

here = Path.cwd()
wanted = ["01_hearing_fly_song.ipynb", "02_labelling_and_training.ipynb",
          "03_looking_at_your_model.ipynb", "Dmel_male.wav"]
missing = [f for f in wanted if not (here / f).exists()]
record("workshop files", True if not missing else None,
       f"all {len(wanted)} are here, in {here.name}/" if not missing
       else f"{len(missing)} of {len(wanted)} not here yet, in {here.name}/")
if missing:
    print("      Nothing to do if you have not been sent the full folder yet.")
    print("      If you do have it, you started JupyterLab above it rather than inside it.")

try:
    probe = here / ".write_probe"
    probe.write_text("ok")
    probe.unlink()
    record("can write here", True, "yes")
except Exception as exc:
    record("can write here", False, f"no ({type(exc).__name__})",
           "move the folder somewhere you own, such as your Documents folder")

In [ ]:
# 9. Part 1 downloads its example recording, so the network has to reach GitHub.
import socket
import urllib.request

url = "https://github.com/janclemenslab/das_unsupervised/releases/download/v0.4/flies.npz"
try:
    socket.setdefaulttimeout(30)
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req) as r:
        size = r.headers.get("Content-Length")
    record("can reach the data", True,
           f"yes ({int(size)/1e6:.1f} MB to download)" if size else "yes")
except Exception as exc:
    record("can reach the data", False, f"no ({type(exc).__name__})",
           "check your connection. On conference wifi, do this before the session")

## Verdict

In [ ]:
# 10. The verdict.
essential   = ["python version", "numpy", "scipy", "matplotlib",
               "can write here", "can reach the data"]
listening   = ["audio widget"]
part1_extra = ["umap", "hdbscan"]
part23      = ["das", "tensorflow", "pandas", "gui toolkit"]
notes       = ["workshop files"]
expected    = essential + listening + part1_extra + part23 + notes

# A check that crashed never gets as far as record(), so it leaves nothing behind. A missing
# result has to count as a problem too: without this line, a cell that crashed would look
# exactly like a cell that passed.
never_ran = [n for n in expected if n not in results]

def failed(group):
    return [n for n in group if n in results and results[n][0] is False]

bad_essential = failed(essential)
bad_listen    = failed(listening)
bad_p1        = failed(part1_extra)
bad_p23       = failed(part23)
pending       = [n for n in notes if n in results and results[n][0] is None]

print("=" * 62)
if not (never_ran or bad_essential or bad_listen or bad_p1 or bad_p23):
    print("READY. Everything the workshop needs is installed and working.")
else:
    if never_ran:
        print(f"INCOMPLETE. These checks never reported: {', '.join(never_ran)}")
        print("  Either you have not run every cell, or one of them stopped with an error.")
        print("  Run them all from the top, fix any red traceback, then read this cell again.")
    if bad_essential:
        print(f"NOT READY. Part 1 will not run: {', '.join(bad_essential)}")
    elif not never_ran:
        print("Part 1 will run.")
    if bad_listen:
        print(f"  Part 1's audio players will not appear: {', '.join(bad_listen)}")
    if bad_p1:
        print(f"  Part 1's final clustering section will skip itself: {', '.join(bad_p1)}")
    if bad_p23:
        print(f"  Parts 2 and 3 will not run: {', '.join(bad_p23)}")
    print("\n  Each line above prints its own fix. Re-run this notebook after fixing.")
if pending:
    print("\n  The workshop notebooks are not in this folder. If you were sent only the setup")
    print("  check, that is expected: they come with the full folder, and nothing checked above")
    print("  depends on them. If you already have that folder, start JupyterLab inside it.")
print("=" * 62)

## If something failed

| What failed | What to do |
|---|---|
| `python version`, or the kernel path looks wrong | **Kernel &rarr; Change Kernel...** and pick the environment you installed. Almost every `No module named ...` error comes from this. |
| `numpy` / `scipy` / `matplotlib` | You are on the wrong kernel, or the environment was never created. Re-read **Install** in `README.md`. |
| `umap` / `hdbscan` | Optional. Only part 1's final clustering section needs them: `pip install umap-learn hdbscan`. Everything else still runs. |
| `das`, `tensorflow` or `pandas` | Parts 2 and 3 need these. Create the DAS environment: follow the **Install** steps in `README.md`, which pin `das==0.32.11`. |
| `gui toolkit` | Part 2 opens the DAS window, which comes from `xarray-behave`. Install it by name, the way install step 3 does, or check that you are on the right kernel. |
| `audio widget` did not render | Open the notebook in **JupyterLab in a browser**. Some editors do not render audio players. |
| You see the player but hear nothing | Not a setup problem. 150 Hz is near the bottom of what laptop speakers manage, so use headphones. Part 1 explains this. |
| `workshop files` listed as *not here yet* | Expected if you were sent only the setup check. The three parts and the recording arrive with the full folder before the session. If you already have that folder, you started JupyterLab above it rather than inside it. |
| `can reach the data` | Part 1 downloads a 2 MB example recording. Do this on a good connection before the session rather than on conference wifi. |
| A check is listed as *never reported* | That cell stopped with an error instead of finishing. Scroll up to the red traceback and fix that first. The verdict cannot tell you anything about a check that never ran. |

### Notes for Windows

- Use **Anaconda Prompt**, not PowerShell or the old Command Prompt. In PowerShell,
  `conda activate` usually fails until you have run `conda init powershell` once and opened
  a new window.
- If Windows Defender or a corporate proxy blocks the download in check 9, it will block
  part 1 too. Sort that out before the session.

Still stuck? The official DAS install page is
<https://janclemenslab.org/das/installation.html>.